## JAX Utilities — Imports & Numerical Safety

Small, dependency-light helpers in **JAX** (`jax.numpy as jnp`) mirroring what shows up in the exam/solutions:
- Stable linear algebra (Cholesky-based solves, log-det)
- Common link functions (sigmoid, softmax, logsumexp)
- Gaussian identities (log pdf, conditioning, BLR posterior/predictive)
- GP utilities (SE kernel, marginal likelihood, predictive)
- Discrete/MC estimators, classification helpers, credibility intervals
- Minimal Metropolis–Hastings (JAX RNG)

Tip: everything returns `jnp` arrays/scalars; add tiny `jitter` before SPD ops.

In [1]:
from typing import Tuple

import jax
import jax.numpy as jnp
from jax import random
from jax.scipy.special import erf, ndtr, ndtri  # Φ, Φ^{-1}
# jax.numpy.linalg has cholesky/solve/etc.

EPS = 1e-12
JITTER = 1e-8

### 1) Numerical Helpers
- `add_jitter`: add tiny diagonal for SPD matrices (covariances)
- `symmetrize`: enforce numerical symmetry

In [2]:
def add_jitter(Sigma: jnp.ndarray, jitter: float = JITTER) -> jnp.ndarray:
    Sigma = jnp.asarray(Sigma, dtype=jnp.float32)
    return Sigma + jitter * jnp.eye(Sigma.shape[0], dtype=Sigma.dtype)

def symmetrize(A: jnp.ndarray) -> jnp.ndarray:
    A = jnp.asarray(A, dtype=jnp.float32)
    return 0.5 * (A + A.T)

### 2) Core Linear Algebra
- `mm`: matrix multiplication
- `inv`, `pinv`: inverses (use sparingly; prefer solves)
- `solve`: generic linear solve
- `cholesky`: Cholesky factor (lower by default)
- `solve_spd`: SPD solve via Cholesky (stable)
- `logdet_spd`: log-determinant via Cholesky
- `trace`: matrix trace

In [3]:
def mm(A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    return jnp.asarray(A) @ jnp.asarray(B)

def inv(A: jnp.ndarray) -> jnp.ndarray:
    return jnp.linalg.inv(jnp.asarray(A))

def pinv(A: jnp.ndarray) -> jnp.ndarray:
    return jnp.linalg.pinv(jnp.asarray(A))

def solve(A: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    return jnp.linalg.solve(jnp.asarray(A), jnp.asarray(b))

def cholesky(A: jnp.ndarray, lower: bool = True) -> jnp.ndarray:
    L = jnp.linalg.cholesky(add_jitter(symmetrize(A)))
    return L if lower else L.T

def solve_spd(A: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    """Solve Ax=b with SPD A using Cholesky (more stable than inv)."""
    L = cholesky(A, lower=True)
    y = jnp.linalg.solve(L, b)
    x = jnp.linalg.solve(L.T, y)
    return x

def logdet_spd(A: jnp.ndarray) -> jnp.ndarray:
    L = cholesky(A, lower=True)
    return 2.0 * jnp.sum(jnp.log(jnp.diag(L)))

def trace(A: jnp.ndarray) -> jnp.ndarray:
    return jnp.trace(jnp.asarray(A))

### 3) Probability Links & Stable Logs
- `sigmoid`, `softmax`: numerically stable variants
- `logsumexp`: stable log-sum-exp
- `std_normal_cdf`: Φ(x) (alias of `ndtr` from `jax.scipy.special`)

In [4]:
def sigmoid(x: jnp.ndarray) -> jnp.ndarray:
    x = jnp.asarray(x)
    return 0.5 * (jnp.tanh(0.5 * x) + 1.0)  # stable

def softmax(z: jnp.ndarray, axis: int = -1) -> jnp.ndarray:
    z = jnp.asarray(z)
    zmax = jnp.max(z, axis=axis, keepdims=True)
    e = jnp.exp(z - zmax)
    return e / (jnp.sum(e, axis=axis, keepdims=True) + EPS)

def logsumexp(a: jnp.ndarray, axis=None, keepdims: bool=False) -> jnp.ndarray:
    a = jnp.asarray(a)
    m = jnp.max(a, axis=axis, keepdims=True)
    out = jnp.log(jnp.sum(jnp.exp(a - m), axis=axis, keepdims=True)) + m
    return out if keepdims else jnp.squeeze(out, axis=axis)

def std_normal_cdf(x: jnp.ndarray) -> jnp.ndarray:
    return ndtr(x)

### 4) Gaussian Identities
- `log_gaussian_1d`, `log_gaussian`: log pdfs
- `gaussian_condition`: parameters of conditional of a partitioned Gaussian
- `gaussian_linreg_posterior`: BLR posterior (conjugate)
- `gaussian_linreg_predict`: BLR posterior predictive for y*

In [5]:
def log_gaussian_1d(x, mu, var) -> jnp.ndarray:
    x = jnp.asarray(x); mu = jnp.asarray(mu); var = jnp.asarray(var)
    return -0.5 * (jnp.log(2.0 * jnp.pi * var) + ((x - mu) ** 2) / var)

def log_gaussian(x: jnp.ndarray, mu: jnp.ndarray, Sigma: jnp.ndarray) -> jnp.ndarray:
    x = jnp.atleast_1d(x); mu = jnp.atleast_1d(mu); Sigma = jnp.asarray(Sigma)
    xc = x - mu
    alpha = solve_spd(Sigma, xc)
    d = x.shape[0]
    return -0.5 * (d * jnp.log(2.0 * jnp.pi) + logdet_spd(Sigma) + xc @ alpha)

def gaussian_condition(
    mu_a: jnp.ndarray, mu_b: jnp.ndarray,
    Sigma_aa: jnp.ndarray, Sigma_ab: jnp.ndarray, Sigma_bb: jnp.ndarray,
    x_b: jnp.ndarray
) -> Tuple[jnp.ndarray, jnp.ndarray]:
    Sigma_bb = add_jitter(Sigma_bb)
    mu_cond = mu_a + Sigma_ab @ solve_spd(Sigma_bb, (x_b - mu_b))
    Sigma_cond = Sigma_aa - Sigma_ab @ solve_spd(Sigma_bb, Sigma_ab.T)
    return mu_cond, symmetrize(Sigma_cond)

def gaussian_linreg_posterior(Phi: jnp.ndarray, y: jnp.ndarray, sigma: float, alpha: float):
    Phi = jnp.asarray(Phi); y = jnp.asarray(y).reshape(-1, 1)
    D = Phi.shape[1]
    S_inv = alpha * jnp.eye(D) + (1.0 / (sigma ** 2)) * (Phi.T @ Phi)
    S = jnp.linalg.inv(S_inv)
    m = (1.0 / (sigma ** 2)) * (S @ Phi.T @ y)
    return m.ravel(), symmetrize(S)

def gaussian_linreg_predict(phi_star: jnp.ndarray, m: jnp.ndarray, S: jnp.ndarray, sigma: float):
    phi_star = jnp.asarray(phi_star).reshape(-1, 1)
    m = jnp.asarray(m).reshape(-1, 1); S = jnp.asarray(S)
    mean = float((m.T @ phi_star).squeeze())
    var = float((phi_star.T @ S @ phi_star).squeeze() + sigma ** 2)
    return mean, var

### 5) Gaussian Processes (SE Kernel)
- `pairwise_dists`: fast pairwise Euclidean distances
- `kernel_matrix`: SE kernel
- `gp_log_marginal_likelihood`: log p(y|θ)
- `gp_predict_f`, `gp_predict_y`: posterior for f*, predictive for y*

In [6]:
def pairwise_dists(X1: jnp.ndarray, X2: jnp.ndarray) -> jnp.ndarray:
    X1 = jnp.atleast_2d(X1); X2 = jnp.atleast_2d(X2)
    X1_sq = jnp.sum(X1 ** 2, axis=1, keepdims=True)
    X2_sq = jnp.sum(X2 ** 2, axis=1, keepdims=True).T
    d2 = jnp.maximum(X1_sq + X2_sq - 2.0 * (X1 @ X2.T), 0.0)
    return jnp.sqrt(d2)

def squared_exponential(dist: jnp.ndarray, kappa: float, lengthscale: float) -> jnp.ndarray:
    return (kappa ** 2) * jnp.exp(-0.5 * (dist ** 2) / (lengthscale ** 2))

def kernel_matrix(X1: jnp.ndarray, X2: jnp.ndarray, kappa: float, lengthscale: float, jitter: float=0.0):
    D = pairwise_dists(X1, X2)
    K = squared_exponential(D, kappa, lengthscale)
    if jitter > 0 and X1.shape[0] == X2.shape[0] and jnp.allclose(X1, X2):
        K = K + jitter * jnp.eye(len(X1), dtype=K.dtype)
    return K

def gp_log_marginal_likelihood(X, y, kappa: float, lengthscale: float, sigma: float) -> jnp.ndarray:
    X = jnp.atleast_2d(X); y = jnp.asarray(y).reshape(-1, 1)
    K = kernel_matrix(X, X, kappa, lengthscale)
    C = add_jitter(K + (sigma ** 2) * jnp.eye(len(X)))
    L = jnp.linalg.cholesky(C)
    alpha = jnp.linalg.solve(L.T, jnp.linalg.solve(L, y))
    term1 = (y.T @ alpha).squeeze()
    term2 = 2.0 * jnp.sum(jnp.log(jnp.diag(L)))
    n = y.shape[0]
    return -0.5 * (term1 + term2 + n * jnp.log(2.0 * jnp.pi))

def gp_predict_f(X_train, y_train, X_star, kappa: float, lengthscale: float, sigma: float):
    X = jnp.atleast_2d(X_train); y = jnp.asarray(y_train).reshape(-1, 1)
    Xs = jnp.atleast_2d(X_star)

    K = kernel_matrix(X, X, kappa, lengthscale)
    Kss = kernel_matrix(Xs, Xs, kappa, lengthscale)
    Ks = kernel_matrix(Xs, X, kappa, lengthscale)

    C = add_jitter(K + (sigma ** 2) * jnp.eye(len(X)))
    L = jnp.linalg.cholesky(C)
    v = jnp.linalg.solve(L, Ks.T)
    alpha = jnp.linalg.solve(L.T, v)
    mean = (alpha.T @ y)
    cov = Kss - (alpha.T @ Ks.T)
    return mean, symmetrize(cov)

def gp_predict_y(X_train, y_train, X_star, kappa: float, lengthscale: float, sigma: float):
    mean_f, cov_f = gp_predict_f(X_train, y_train, X_star, kappa, lengthscale, sigma)
    cov_y = cov_f + (sigma ** 2) * jnp.eye(cov_f.shape[0], dtype=cov_f.dtype)
    return mean_f, cov_y

### 6) Discrete & Monte Carlo Estimators
- `discrete_expectation`, `discrete_variance`: grid approximations (like exam Part 2)
- `mc_expectation`: MC mean and MC standard error

In [7]:
def discrete_expectation(values: jnp.ndarray, probs: jnp.ndarray) -> jnp.ndarray:
    values = jnp.asarray(values, dtype=jnp.float32)
    probs = jnp.asarray(probs, dtype=jnp.float32)
    probs = probs / (jnp.sum(probs) + EPS)
    return jnp.sum(values * probs, axis=-1)

def discrete_variance(values: jnp.ndarray, probs: jnp.ndarray) -> jnp.ndarray:
    mu = discrete_expectation(values, probs)
    values = jnp.asarray(values, dtype=jnp.float32)
    probs = jnp.asarray(probs, dtype=jnp.float32)
    probs = probs / (jnp.sum(probs) + EPS)
    return jnp.sum(((values - mu) ** 2) * probs, axis=-1)

def mc_expectation(f_vals: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    f_vals = jnp.asarray(f_vals).ravel()
    mean = jnp.mean(f_vals)
    se = jnp.std(f_vals, ddof=1) / jnp.sqrt(f_vals.shape[0])
    return mean, se

### 7) Classification Shortcuts
- `plugin_predict_binary`: logistic plug-in (MAP)
- `probit_approx_from_gaussian_f`: probit approximation for GP/logistic
- `entropy_categorical`, `confidence_categorical`: for predictive summaries

In [8]:
def plugin_predict_binary(phi_star: jnp.ndarray, w_map: jnp.ndarray) -> float:
    return float(sigmoid(jnp.dot(w_map, phi_star)))

def probit_approx_from_gaussian_f(mean_f: float, var_f: float) -> float:
    # p ≈ Φ( μ / sqrt(1 + π var / 8) )
    denom = jnp.sqrt(1.0 + jnp.pi * var_f / 8.0)
    return float(ndtr(mean_f / denom))

def entropy_categorical(p: jnp.ndarray) -> float:
    p = jnp.asarray(p, dtype=jnp.float32)
    p = p / (jnp.sum(p) + EPS)
    mask = p > 0
    return float(-jnp.sum(p[mask] * jnp.log(p[mask])))

def confidence_categorical(p: jnp.ndarray) -> float:
    p = jnp.asarray(p, dtype=jnp.float32)
    p = p / (jnp.sum(p) + EPS)
    return float(jnp.max(p))

### 8) Gaussian Credibility Intervals
- `gaussian_ci`: mean ± z * sqrt(var), using `ndtri` (Φ⁻¹)

In [9]:
def gaussian_ci(mean: float, var: float, level: float = 0.95) -> Tuple[float, float]:
    alpha = 1.0 - level
    z = ndtri(1.0 - alpha / 2.0)
    halfwidth = float(z * jnp.sqrt(var))
    return (float(mean) - halfwidth, float(mean) + halfwidth)

### 9) Minimal Metropolis–Hastings (Random-Walk)
- Gaussian proposal: x' = x + N(0, prop_std² I)
- Pure JAX RNG; Python loop for clarity (fine for exam-scale tasks)

In [10]:
def metropolis_hastings(
    log_target,                # function: x -> log p(x | data)
    x0: jnp.ndarray,
    key: jax.random.PRNGKey,
    prop_std: float = 0.5,
    n_samples: int = 10_000,
    burn_in: int = 1_000,
    thin: int = 1
) -> jnp.ndarray:
    x = jnp.asarray(x0, dtype=jnp.float32).ravel()
    d = x.shape[0]
    samples = []
    logp = float(log_target(x))
    k = key
    for t in range(n_samples):
        k, k_step = random.split(k)
        proposal = x + prop_std * random.normal(k_step, shape=(d,))
        logp_prop = float(log_target(proposal))
        k, k_u = random.split(k)
        if jnp.log(random.uniform(k_u)) < (logp_prop - logp):
            x = proposal
            logp = logp_prop
        samples.append(x)
    samples = jnp.stack(samples, axis=0)
    return samples[burn_in::thin]

### 10) Design Matrix Shortcuts
- `polynomial_design_matrix`: `[1, x, x², …, x^degree]`

In [11]:
def polynomial_design_matrix(x: jnp.ndarray, degree: int) -> jnp.ndarray:
    x = jnp.asarray(x).ravel()
    cols = [jnp.ones_like(x)]
    for k in range(1, degree + 1):
        cols.append(x ** k)
    return jnp.stack(cols, axis=1)